<!-- SPDX-License-Identifier: Apache-2.0 -->
# UAIRT: one example per backend

This notebook builds [UAIRT](https://github.com/vnk8071/unified-ai-runtime) and runs a tiny model through each backend it can, using
the Python binding. Every section **skips itself with a reason** when its runtime is missing, so you can run the whole notebook
top to bottom on any machine and read the summary at the end.

| Backend | Model it loads | What you install yourself | Section |
|---|---|---|---|
| `reference` | none (copies its input) | nothing, built in | 1 |
| `onnxruntime` | `.onnx` | `pip install onnxruntime` (the next cell does it) | 2 |
| `ncnn` | `.param` + `.bin` | nothing: a pinned submodule | 3 |
| `executorch` | `.pte` | an ExecuTorch install (`EXECUTORCH_ROOT`), `torch` and `executorch` for export | 4 |
| `whispercpp` | ggml Whisper `.bin` | nothing but a model file: a pinned submodule | 5 |
| `llamacpp` | `.gguf` | nothing but a model file: a pinned submodule | 6 |
| `coreml` | `.mlpackage` | macOS with Xcode; `coremltools` to make a model (the next cell installs it) | 7 |
| `openvino` | `.xml` + `.bin`, `.onnx` | `pip install openvino` (the next cell does it) | 8 |
| `tflite` | `.tflite` | a TFLite C library (`TFLITE_LIBRARY`) | 9 |
| `qnn` | `.dlc`, `.bin` | Qualcomm's QAIRT SDK (`QNN_SDK_ROOT`), a Snapdragon device for the NPU | 10 |
| `tensorrt` | `.engine` | an NVIDIA GPU, CUDA, TensorRT: see `uairt_tensorrt_colab.ipynb` | 11 |

Vendor SDKs (QAIRT, TensorRT, Xcode) are never part of this repository: you download them and accept their licences. Which
backends have been run on real hardware, and on what, is in the repository's `README.md` and `docs/api.md`.

## Install what pip provides

Some backends need nothing from you beyond a wheel. This cell installs those that have one, if they are missing:

| Wheel | What it enables |
|---|---|
| `numpy` | the Python binding |
| `onnxruntime` | the `onnxruntime` backend (its runtime library; the header is fetched by the setup cell) |
| `openvino` | the `openvino` backend (headers and libraries) |
| `coremltools` | the `coreml` section (macOS only): it makes the demo model; the backend itself is a system framework |

It uses `pip`, or `uv pip` when the environment has no pip (a `uv venv` has none). Set `INSTALL_FROM_PIP = False` to leave your
environment alone. The rest cannot come from a wheel: `ncnn`, `whispercpp` and `llamacpp` are built from the repository's submodules,
`executorch` needs a C++ install (`EXECUTORCH_ROOT`), `tflite` and `qnn` need their vendor SDKs, and `tensorrt` has its own notebook.

In [ ]:
import importlib
import importlib.util
import platform
import shutil
import subprocess
import sys

INSTALL_FROM_PIP = True        # False: change nothing in this environment
PIP_PACKAGES = ["numpy", "onnxruntime", "openvino"] + (["coremltools"] if platform.system() == "Darwin" else [])

def pip_install(packages):
    missing = [name for name in packages if importlib.util.find_spec(name) is None]
    if not missing:
        print("already installed:", ", ".join(packages))
        return
    attempts = [[sys.executable, "-m", "pip", "install", "-q", *missing]]
    if shutil.which("uv"):
        attempts.append(["uv", "pip", "install", "-q", "--python", sys.executable, *missing])
    for command in attempts:
        done = subprocess.run(command, capture_output=True, text=True)
        if done.returncode == 0:
            importlib.invalidate_caches()
            print("installed:", ", ".join(missing), "  (with", " ".join(command[:3]) + ")")
            return
    print("could not install", ", ".join(missing), "- the sections that need them will skip.\n", done.stderr.strip()[-400:])

print("python", sys.version.split()[0], "|", sys.executable)
if INSTALL_FROM_PIP:
    pip_install(PIP_PACKAGES)

## Setup

Edit `BACKENDS` if you want to force a choice; by default a backend is built when what it needs is present: `ncnn` and `whispercpp` always,
`onnxruntime` and `openvino` when they are `pip install`ed, `coreml` on a Mac, the others when their `*_ROOT` variable is set.
The first build compiles NCNN and whisper.cpp and takes several minutes.

In [ ]:
import importlib.util
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

# ---- Configure --------------------------------------------------------------------------------------------------
# Where the build goes. Delete it to start over.
BUILD_DIR = Path(os.environ.get("UAIRT_NB_BUILD", "build-notebook")).resolve()
# A checkout of the repository. None: look upward from here, and clone it if there is none.
REPO = os.environ.get("UAIRT_REPO")
# None: choose automatically (see below). Or a list, for example ["ncnn", "onnxruntime"].
BACKENDS = None
# Set to True to download ggml-tiny.en.bin (78 MB, from huggingface.co/ggerganov/whisper.cpp) for section 5.
DOWNLOAD_WHISPER_MODEL = False
# ------------------------------------------------------------------------------------------------------------------

def find_repo():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "include" / "uairt" / "uairt.h").exists():
            return base
    return None

REPO = Path(REPO).resolve() if REPO else find_repo()
if REPO is None:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/vnk8071/unified-ai-runtime.git",
                    "unified-ai-runtime"], check=True)
    REPO = Path("unified-ai-runtime").resolve()
print("repository:", REPO)

CMAKE_OPTION = {
    "onnxruntime": "UAIRT_BUILD_ONNXRUNTIME", "ncnn": "UAIRT_BUILD_NCNN", "executorch": "UAIRT_BUILD_EXECUTORCH",
    "whispercpp": "UAIRT_BUILD_WHISPERCPP", "llamacpp": "UAIRT_BUILD_LLAMACPP", "coreml": "UAIRT_BUILD_COREML",
    "openvino": "UAIRT_BUILD_OPENVINO", "tflite": "UAIRT_BUILD_TFLITE", "qnn": "UAIRT_BUILD_QNN",
    "tensorrt": "UAIRT_BUILD_TENSORRT",
}
SUBMODULE = {"ncnn": "third_party/ncnn", "whispercpp": "third_party/whisper.cpp", "llamacpp": "third_party/llama.cpp"}

def pip_package_dir(name):
    # Where a pip-installed package lives, found without importing it (OpenVINO's import starts a telemetry subprocess).
    spec = importlib.util.find_spec(name)
    return Path(spec.submodule_search_locations[0]) if spec and spec.submodule_search_locations else None

def auto_backends():
    chosen = ["ncnn", "whispercpp"]                       # pinned submodules: nothing to install
    # ONNX Runtime: the plugin needs only its header (fetched below) and loads the pip-installed library at run time.
    if os.environ.get("ONNXRUNTIME_ROOT") or pip_package_dir("onnxruntime"):
        chosen.append("onnxruntime")
    if os.environ.get("EXECUTORCH_ROOT"):
        chosen.append("executorch")
    if platform.system() == "Darwin":
        chosen.append("coreml")
    # OpenVINO: `pip install openvino` ships the C headers and libraries the plugin needs.
    openvino_dir = pip_package_dir("openvino")
    if not os.environ.get("OPENVINO_ROOT") and openvino_dir and (openvino_dir / "include" / "openvino" / "c" / "openvino.h").exists():
        os.environ["OPENVINO_ROOT"] = str(openvino_dir)
    if os.environ.get("OPENVINO_ROOT"):
        chosen.append("openvino")
    if os.environ.get("UAIRT_LLAMACPP_MODEL"):
        chosen.append("llamacpp")
    if os.environ.get("TFLITE_LIBRARY") and os.environ.get("TFLITE_INCLUDE_DIR"):
        chosen.append("tflite")
    if os.environ.get("QNN_SDK_ROOT"):
        chosen.append("qnn")
    if shutil.which("nvidia-smi") and os.environ.get("TENSORRT_ROOT"):
        chosen.append("tensorrt")
    return chosen

BACKENDS = BACKENDS if BACKENDS is not None else auto_backends()
print("building:", BACKENDS)

In [ ]:
def run_logged(command):
    # Runs a command and prints about one progress line per 10%, so a long build does not look hung. The full output is
    # kept and shown if the command fails.
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    log, shown = [], -10
    for line in process.stdout:
        log.append(line)
        progress = re.match(r"\[\s*(\d+)%\]", line)
        if progress and int(progress.group(1)) >= shown + 10:
            shown = int(progress.group(1))
            print(line.rstrip()[:120], flush=True)
    if process.wait():
        print("".join(log)[-4000:])
        raise SystemExit(f"{command[0]} {command[1]} failed; read the output above")
    print("".join(log).strip().splitlines()[-1][:120])

# Fetch the pinned submodules the chosen backends use (skipped when already present).
for name in BACKENDS:
    path = SUBMODULE.get(name)
    if path and not (REPO / path / "CMakeLists.txt").exists():
        subprocess.run(["git", "-C", str(REPO), "submodule", "update", "--init", "--depth", "1", path], check=True)

# The ONNX Runtime plugin needs only ONNX Runtime's header. Without ONNXRUNTIME_ROOT, fetch it once into the repository's .ort/
# (git-ignored); the pip-installed library is found at run time.
if "onnxruntime" in BACKENDS and not os.environ.get("ONNXRUNTIME_ROOT") and not (REPO / ".ort" / "include" / "onnxruntime_c_api.h").exists():
    subprocess.run(["bash", str(REPO / "scripts" / "ci" / "fetch_onnxruntime_headers.sh")], check=True)

configure = ["cmake", "-S", str(REPO), "-B", str(BUILD_DIR), "-DBUILD_SHARED_LIBS=ON", "-DCMAKE_BUILD_TYPE=Release",
             "-DCMAKE_COLOR_MAKEFILE=OFF", "-DUAIRT_BUILD_TESTS=OFF", "-DUAIRT_BUILD_EXAMPLES=OFF"] + \
            [f"-D{CMAKE_OPTION[n]}=ON" for n in BACKENDS]
run_logged(configure)
run_logged(["cmake", "--build", str(BUILD_DIR), "-j", str(os.cpu_count() or 2)])

In [ ]:
# Point the Python binding at the library and plugins we just built, then import it.
library = next(p for p in BUILD_DIR.glob("libuairt.*") if p.suffix in (".dylib", ".so", ".dll"))
os.environ["UAIRT_LIBRARY"] = str(library)
os.environ["UAIRT_PLUGIN_PATH"] = str(BUILD_DIR)
sys.path.insert(0, str(REPO / "bindings" / "python"))

import numpy as np
import uairt

# available_backends() lists every name the package knows; usable ones are registered or have a plugin file we can load.
AVAILABLE = {name for name, info in uairt.available_backends().items() if info["registered"] or info["plugin"]}
print("UAIRT", uairt.version(), "| usable backends:", sorted(AVAILABLE))

class Skip(Exception):
    pass

def need(condition, why):
    if not condition:
        raise Skip(why)

RESULTS = {}

def section(name):
    # Runs the decorated function now. A skip or a failure is recorded and printed; the notebook goes on.
    def run(function):
        try:
            function()
            RESULTS[name] = ("ran", "")
        except Skip as skip:
            RESULTS[name] = ("skipped", str(skip))
            print("skipped:", skip)
        except Exception as error:
            RESULTS[name] = ("FAILED", f"{type(error).__name__}: {error}")
            print("FAILED:", RESULTS[name][1])
        return function
    return run

WORK = Path(tempfile.mkdtemp(prefix="uairt-nb-"))

## 1. reference

The test backend built into the library. It copies its one `float32 [1, 4]` input to its output, which makes it the smallest
possible example of the whole call sequence: engine, model, run.

In [ ]:
@section("reference")
def _():
    need("reference" in AVAILABLE, "the reference backend was not built (UAIRT_BUILD_REFERENCE_BACKEND=OFF)")
    with uairt.Engine("reference") as engine, engine.load_model("unused") as model:
        print("inputs :", model.inputs)
        print("outputs:", model.outputs)
        x = np.array([[1, 2, 3, 4]], dtype=np.float32)
        (y,) = model.run(x)
    print(y)
    assert np.array_equal(x, y)

## 2. onnxruntime

Runs `.onnx` models. The plugin needs only ONNX Runtime's header at build time and loads the library at run time. With
`pip install onnxruntime` done before the setup cell, the notebook fetches the header itself (`scripts/ci/fetch_onnxruntime_headers.sh`,
into the git-ignored `.ort/`) and uses the pip library. Or set `ONNXRUNTIME_ROOT` to a release directory. Alternatively
`pip install "uairt[onnxruntime]"` brings a prebuilt plugin and needs no build at all.
`tests/data/add_mul.onnx` takes two inputs and returns their sum and product.

In [ ]:
@section("onnxruntime")
def _():
    need("onnxruntime" in AVAILABLE, "plugin not built: pip install onnxruntime (or set ONNXRUNTIME_ROOT to a release) and re-run from the setup cell")
    x = np.array([[1, 2, 3, 4]], dtype=np.float32)
    y = np.array([[10, 20, 30, 40]], dtype=np.float32)
    with uairt.Engine("onnxruntime", intra_op_threads=1) as engine:
        with engine.load_model(REPO / "tests" / "data" / "add_mul.onnx") as model:
            print("inputs :", [(i.name, i.shape) for i in model.inputs])
            total, product = model.run(x, y)
    print(total, product)
    np.testing.assert_allclose(total, x + y)
    np.testing.assert_allclose(product, x * y)

## 3. ncnn

NCNN's `.param` file records no input shapes, so the engine takes `input_shapes`. NCNN is a pinned submodule and is built into the
plugin. This writes a one-layer fully connected net by hand (a 3-to-4 `InnerProduct`) and checks the result against NumPy;
`fp16=false` because NCNN's default on ARM CPUs computes in half precision and would differ in the fourth digit.

In [ ]:
@section("ncnn")
def _():
    need("ncnn" in AVAILABLE, "plugin not built (git submodule update --init third_party/ncnn)")
    rng = np.random.default_rng(1)
    weights = rng.standard_normal((4, 3)).astype(np.float32)
    bias = rng.standard_normal(4).astype(np.float32)
    model_dir = WORK / "ncnn_fc"
    model_dir.mkdir()
    (model_dir / "fc.param").write_text("7767517\n2 2\nInput in0 0 1 in0\nInnerProduct fc 1 1 in0 out0 0=4 1=1 2=12\n")
    (model_dir / "fc.bin").write_bytes(np.uint32(0).tobytes() + weights.tobytes() + bias.tobytes())

    x = np.array([0.5, -1.0, 2.0], dtype=np.float32)
    with uairt.Engine("ncnn", input_shapes="3", fp16="false") as engine:
        with engine.load_model(model_dir) as model:
            (y,) = model.run(x)
    print(y)
    np.testing.assert_allclose(y, weights @ x + bias, rtol=1e-5, atol=1e-6)

## 4. executorch

Runs ExecuTorch `.pte` programs. You build and install ExecuTorch yourself (`docs/backends/executorch.md` lists the flags) and point
`EXECUTORCH_ROOT` at it. To export a model you also need the `torch` and `executorch` Python packages. Which hardware the model runs
on is decided at export time by the delegate: this one uses XNNPACK (CPU). `num_threads=1` is worth setting for a model that runs in
microseconds, where ExecuTorch's default pool of every core costs more than it saves.

In [ ]:
@section("executorch")
def _():
    need("executorch" in AVAILABLE, "plugin not built: set EXECUTORCH_ROOT to your ExecuTorch install and rebuild")
    try:
        import torch
        from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
        from executorch.exir import to_edge_transform_and_lower
    except ImportError as error:
        raise Skip(f"exporting needs the torch and executorch Python packages ({error})")

    class Net(torch.nn.Module):
        def __init__(self):
            super().__init__()
            torch.manual_seed(0)
            self.first, self.second = torch.nn.Linear(8, 16), torch.nn.Linear(16, 4)

        def forward(self, x):
            return self.second(torch.relu(self.first(x)))

    net = Net().eval()
    exported = torch.export.export(net, (torch.zeros(1, 8),))
    program = to_edge_transform_and_lower(exported, partitioner=[XnnpackPartitioner()]).to_executorch()
    path = WORK / "net.pte"
    path.write_bytes(program.buffer)

    x = np.random.default_rng(0).random((1, 8), dtype=np.float32)
    with uairt.Engine("executorch", num_threads=1) as engine, engine.load_model(path) as model:
        (y,) = model.run(x)
    print(y)
    np.testing.assert_allclose(y, net(torch.from_numpy(x)).detach().numpy(), rtol=1e-4, atol=1e-5)

## 5. whispercpp

Speech to text with a ggml Whisper model. The tensor API has no strings and shapes are static, so the model takes
`audio` (float32 `[n_samples]`, mono 16 kHz) and returns `text` (uint8, UTF-8, zero padded) and `text_length` (int32 `[1]`).
`n_samples` is an engine option: give the real clip length. You need a model file; set `UAIRT_WHISPER_MODEL` to one, or set
`DOWNLOAD_WHISPER_MODEL = True` in the setup cell. The audio is whisper.cpp's own `samples/jfk.wav`.

In [ ]:
@section("whispercpp")
def _():
    need("whispercpp" in AVAILABLE, "plugin not built (git submodule update --init third_party/whisper.cpp)")
    model_path = os.environ.get("UAIRT_WHISPER_MODEL")
    if not model_path and DOWNLOAD_WHISPER_MODEL:
        import urllib.request
        model_path = str(BUILD_DIR / "ggml-tiny.en.bin")
        if not Path(model_path).exists():
            urllib.request.urlretrieve("https://huggingface.co/ggerganov/whisper.cpp/resolve/main/ggml-tiny.en.bin", model_path)
    need(model_path and Path(model_path).exists(),
         "set UAIRT_WHISPER_MODEL to a ggml Whisper model (or DOWNLOAD_WHISPER_MODEL = True)")
    wav = REPO / "third_party" / "whisper.cpp" / "samples" / "jfk.wav"
    need(wav.exists(), "samples/jfk.wav not found (git submodule update --init third_party/whisper.cpp)")

    import wave
    with wave.open(str(wav), "rb") as audio:
        assert (audio.getframerate(), audio.getnchannels(), audio.getsampwidth()) == (16000, 1, 2)
        pcm = np.frombuffer(audio.readframes(audio.getnframes()), dtype=np.int16).astype(np.float32) / 32768.0

    with uairt.Engine("whispercpp", n_samples=len(pcm)) as engine, engine.load_model(model_path) as model:
        text, length = model.run(pcm)
    transcript = bytes(text[: int(length[0])]).decode("utf-8").strip()
    print(transcript)
    assert transcript

## 6. llamacpp

GGUF language models, used through sessions rather than `run`. It needs a model file: set `UAIRT_LLAMACPP_MODEL` before building to
a small GGUF (for example a Q4 or Q8 instruct model from Hugging Face). `n_gpu_layers=0` keeps it on the CPU.

In [ ]:
@section("llamacpp")
def _():
    need("llamacpp" in AVAILABLE, "plugin not built (set UAIRT_LLAMACPP_MODEL, then rebuild)")
    model_path = os.environ.get("UAIRT_LLAMACPP_MODEL")
    need(model_path and Path(model_path).exists(), "set UAIRT_LLAMACPP_MODEL to a GGUF file")
    with uairt.Engine("llamacpp", n_gpu_layers=0) as engine, engine.load_model(model_path) as model:
        pieces = list(model.generate("The capital of France is", max_tokens=16))
    print("".join(pieces))
    assert pieces

## 7. coreml

Apple's Core ML, including the Neural Engine, on macOS. The plugin loads `.mlpackage`, `.mlmodelc` and `.mlmodel`. This builds a
one-operation program with `coremltools` (`y = x + 2`) and runs it on the CPU. `compute_units` is `all`, `cpu_only`,
`cpu_and_gpu` or `cpu_and_ne`.

In [ ]:
@section("coreml")
def _():
    need("coreml" in AVAILABLE, "plugin not built (macOS with Xcode only)")
    try:
        import coremltools as ct
        from coremltools.converters.mil import Builder as mb
    except ImportError as error:
        raise Skip(f"making a model needs coremltools ({error})")

    @mb.program(input_specs=[mb.TensorSpec(shape=(1, 4))])
    def program(x):
        return mb.add(x=x, y=np.float32(2.0), name="y")

    path = WORK / "add2.mlpackage"
    ct.convert(program, convert_to="mlprogram", compute_units=ct.ComputeUnit.CPU_ONLY).save(str(path))
    x = np.array([[1, 2, 3, 4]], dtype=np.float32)
    with uairt.Engine("coreml", compute_units="cpu_only") as engine, engine.load_model(path) as model:
        print("inputs :", [(i.name, i.shape, i.dtype) for i in model.inputs])
        (y,) = model.run(x)
    print(y)
    np.testing.assert_allclose(y, x + 2)

## 8. openvino

Intel's OpenVINO on the CPU, an Intel GPU or an NPU. The plugin loads OpenVINO IR (`.xml` + `.bin`) **or `.onnx` directly**, so no
conversion step is needed here. `pip install openvino` is enough for the CPU: the setup cell finds the installed package
(it holds `include/openvino/c/openvino.h`) by itself; or set `OPENVINO_ROOT`. (We do not call `openvino.convert_model` on purpose: its telemetry starts a
subprocess that can hang inside a Jupyter kernel.)

In [ ]:
@section("openvino")
def _():
    need("openvino" in AVAILABLE, "plugin not built: pip install openvino (or set OPENVINO_ROOT to its package directory) and re-run from the setup cell")
    x = np.array([[1, 2, 3, 4]], dtype=np.float32)
    y = np.array([[10, 20, 30, 40]], dtype=np.float32)
    with uairt.Engine("openvino", device="CPU") as engine:
        with engine.load_model(REPO / "tests" / "data" / "add_mul.onnx") as model:
            print("inputs :", [(i.name, i.shape) for i in model.inputs])
            total, product = model.run(x, y)
    print(total, product)
    np.testing.assert_allclose(total, x + y)
    np.testing.assert_allclose(product, x * y)

## 9. tflite

TensorFlow Lite / LiteRT through its C API, on the CPU or, on Qualcomm hardware, through the QNN delegate. It needs a TFLite C library
built for your target (`TFLITE_LIBRARY`, `TFLITE_INCLUDE_DIR`) and a `.tflite` model (`UAIRT_TFLITE_MODEL`); there is no tiny model to
generate here without TensorFlow.

In [ ]:
@section("tflite")
def _():
    need("tflite" in AVAILABLE, "plugin not built: set TFLITE_LIBRARY and TFLITE_INCLUDE_DIR, then rebuild")
    model_path = os.environ.get("UAIRT_TFLITE_MODEL")
    need(model_path and Path(model_path).exists(), "set UAIRT_TFLITE_MODEL to a .tflite file")
    with uairt.Engine("tflite", num_threads=2) as engine, engine.load_model(model_path) as model:
        inputs = [np.zeros(i.shape, i.dtype) for i in model.inputs]
        outputs = model.run(*inputs)
    print([o.shape for o in outputs])

## 10. qnn

Qualcomm's AI Runtime (QAIRT) for Snapdragon CPUs, GPUs and the Hexagon NPU. You download QAIRT yourself (`QNN_SDK_ROOT`) and accept its
licence; the NPU needs a Snapdragon device and a `.dlc` model made by `qairt-converter`. `AutoModel` picks the backend from the file
and maps `device="npu"` to the QNN options.

In [ ]:
@section("qnn")
def _():
    need("qnn" in AVAILABLE, "plugin not built: set QNN_SDK_ROOT to your QAIRT SDK and rebuild")
    model_path = os.environ.get("UAIRT_QNN_MODEL")
    need(model_path and Path(model_path).exists(), "set UAIRT_QNN_MODEL to a .dlc file")
    with uairt.AutoModel.from_file(model_path, device="npu") as model:
        print(model.backend, model.options)
        outputs = model.run(*[np.zeros(i.shape, i.dtype) for i in model.inputs])
    print([o.shape for o in outputs])

## 11. tensorrt

NVIDIA GPUs, from a serialized engine (`.engine`). An engine is built for one GPU model and one TensorRT version, so build it on the
machine that runs it. Building the plugin needs TensorRT's headers, which the `tensorrt` pip package lacks; the companion notebook
**`uairt_tensorrt_colab.ipynb`** does the whole thing on a free Colab GPU: install, build, export a model, build an engine, run it and
compare with PyTorch. This cell only runs an engine you already have, in `UAIRT_TRT_ENGINE`.

In [ ]:
@section("tensorrt")
def _():
    need("tensorrt" in AVAILABLE, "plugin not built: needs an NVIDIA GPU, CUDA and TensorRT (see uairt_tensorrt_colab.ipynb)")
    engine_path = os.environ.get("UAIRT_TRT_ENGINE")
    need(engine_path and Path(engine_path).exists(), "set UAIRT_TRT_ENGINE to a TensorRT .engine file")
    with uairt.Engine("tensorrt") as engine, engine.load_model(engine_path) as model:
        outputs = model.run(*[np.zeros(i.shape, i.dtype) for i in model.inputs])
    print([o.shape for o in outputs])

## Summary

In [ ]:
width = max(len(name) for name in RESULTS)
for name, (status, detail) in RESULTS.items():
    print(f"{name:<{width}}  {status:<8}  {detail}")
failed = [name for name, (status, _) in RESULTS.items() if status == "FAILED"]
assert not failed, f"failed: {failed}"

## Where to go next

- `docs/api.md` is the C API and the options each backend takes; `bindings/python/README.md` covers `AutoModel`, buffers and sessions.
- `docs/agents/export-model.md` says how to get a PyTorch or Hugging Face model into each format.
- `docs/backends/<name>.md` has the install steps, limits and the hardware each backend was verified on.